# GreenSite AI — Baghdad Greening MVP (Google Colab)

**Goal.** Read the raw Baghdad satellite/climate/soil CSV (one row per 1 km cell per month) and produce a ranked table of cells that may deserve a **closer look** for vegetation restoration or greening.

The notebook has **three separate parts**. They are kept apart on purpose:

| Part | What it is | Section |
|---|---|---|
| **K-Means clustering** | Machine learning, *unsupervised*: groups cells with similar vegetation history. Exploratory only. | 6 |
| **Rule-based screening** | Transparent, adjustable rules: (a) *may intervention be needed?* (b) *does the soil look more or less promising?* | 7, 8 |
| **Ranking** | Combines need first, then potential, then data confidence. K-Means is **not** used here. | 9 |

**What this notebook does NOT do**

* There are **no labels** of successful or failed planting, so nothing here predicts planting success, and no fake "suitable = 1/0" column exists.
* K-Means is **not** a predictive model and has no accuracy score. Its clusters are shown next to the ranking only for context.
* Low NDVI alone does **not** prove degradation, and a "promising" soil rating does **not** mean a site can be planted.
* The output means *"worth a closer look"*, never *"plant here"*.

**How to run:** `Runtime → Run all`. The only manual step is choosing the CSV file in Section 2. The raw CSV is never modified.

**Key terms**

| Term | Meaning |
|---|---|
| **NDVI** | A greenness index from satellites, from -1 to 1. Higher usually means more green vegetation. |
| **QA flag** | A quality code attached to each NDVI value (0 good, 1 marginal, 2 snow/ice, 3 cloudy). |
| **Cell** | One 1 km square on the map, identified by `cell_id`. |
| **Seasonal anomaly** | NDVI minus the same cell's usual NDVI in the same calendar month. Removes the normal summer/winter swing. |
| **K-Means** | An algorithm that splits items into *k* groups by similarity. It needs no labels. |
| **Silhouette score** | A number from -1 to 1 that says how well separated the K-Means groups are. |
| **Screening** | Applying simple rules to decide which cells to look at first. It is not a diagnosis. |

## 1. Imports and settings

All libraries are imported once, here. All adjustable numbers are also collected in one cell, so you can change a threshold and re-run without searching the notebook. **Every threshold is an MVP assumption, not an ecological standard.**

In [ ]:
# Import the libraries used in this notebook.
# pandas and numpy handle the data, matplotlib draws the plots, scikit-learn provides K-Means.
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display               # display() shows a DataFrame as a formatted table
from google.colab import files                    # Colab tool for uploading and downloading files
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score      # measures how well separated the clusters are
from sklearn.preprocessing import StandardScaler  # puts features on a comparable scale before clustering

In [ ]:
# Collect every adjustable number in one place (change here, then re-run from this cell downward).
RANDOM_SEED = 42                  # fixed seed: K-Means and the silhouette sample give the same answer every run

MIN_VALID_MONTHS = 8              # need this many reliable months (out of 12) to trust a 12-month average
N_CLUSTERS = 3                    # K-Means starts with 3 groups. This is an initial choice and is NOT tuned
SILHOUETTE_SAMPLE_SIZE = 3000     # silhouette is computed on a random sample of cells (faster)

LOW_NDVI_PERCENTILE = 25          # lowest 25% of cells count as "low vegetation" (higher = more cells flagged)
DECLINE_THRESHOLD = -0.03         # drop in 12-month mean NDVI that counts as "declining" (closer to 0 = more flagged)
SEASONAL_DROP_THRESHOLD = -0.03   # drop versus the usual same-month level (closer to 0 = more flagged)
MIN_USABLE_INDICATORS = 2         # need at least this many of the 3 NDVI indicators, else "Insufficient evidence"

SAND_UPPER_QUANTILE = 0.75        # sand share above this city-wide quantile counts as "sandy" (greening potential)
TOP_N_CANDIDATES = 100            # how many top candidates are exported

pd.options.display.float_format = "{:.3f}".format   # show 3 decimals in tables
pd.options.display.width = 200                      # allow wider tables in the output

## 2. Data upload and loading

**Upload the ORIGINAL raw CSV** (about 54 MB, so wait for the upload bar to finish).

* `df_raw` is the original data and is **never modified**.
* `df` is the working copy used everywhere else.
* A *fingerprint* (hash) of `df_raw` is saved, so Section 12 can prove the raw data did not change.

In [ ]:
# Upload the raw CSV and keep one untouched copy.
# Working on a copy means we can always prove that the original data was not changed.
uploaded = files.upload()                          # opens the Colab upload dialog
csv_name = list(uploaded.keys())[0]                # name of the file that was uploaded

df_raw = pd.read_csv(csv_name)                     # original data: never modified
df = df_raw.copy()                                 # working copy

# hash_pandas_object turns every row into a number; the sum is a fingerprint of the whole table.
raw_fingerprint = pd.util.hash_pandas_object(df_raw, index=False).sum()
print("Loaded file:", csv_name)

## 3. Initial data inspection and validation

This section is **data-quality validation**: we check the facts instead of trusting what we were told. Nothing is changed except the `date` column type and the row order.

**3.1 Check the columns.** The notebook stops with a clear message if a required column is missing.

*Expected output:* `Missing required columns: none`.

In [ ]:
# Check that every column we need exists in the uploaded file.
# A missing column would cause confusing errors later, so we stop early with a clear message.
required_columns = ["cell_id", "latitude", "longitude", "date", "ndvi_raw", "ndvi_summary_qa", "temperature_2m_K",
                    "dewpoint_2m_K", "humidity_rh2m_pct", "total_precipitation_m", "soil_moisture_vwc_layer1",
                    "total_evaporation_m", "potential_evaporation_m", "soil_ph_x10", "soil_organic_carbon_dg_kg",
                    "sand_g_kg", "clay_g_kg", "land_cover_code", "power_lat", "power_lon"]
missing_columns = [column for column in required_columns if column not in df.columns]
print("Missing required columns:", missing_columns or "none")
if missing_columns:
    raise ValueError(f"Required column(s) missing: {missing_columns}. Check the file you uploaded.")

**3.2 Basic facts and missing values.** We print the size, the dates, the number of cells, duplicates and missing values.

*Expected output:* a monthly date range, one row per cell and month (no duplicates), and missing values **only in the soil columns**. Missing soil is common in built-up areas, so those cells are kept.

In [ ]:
# Print the basic facts about the file: size, dates, cells, duplicates and missing values.
# These numbers confirm that the file matches what we expect before any analysis starts.
df["date"] = pd.to_datetime(df["date"])           # convert text dates into real dates

print("Shape (rows, columns):", df.shape)
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("Unique cells:", df["cell_id"].nunique())
print("Duplicate (cell_id, date) rows:", df.duplicated(subset=["cell_id", "date"]).sum())
print("Observations per cell (count: number of cells):", df.groupby("cell_id").size().value_counts().to_dict())

important_columns = ["ndvi_raw", "ndvi_summary_qa", "temperature_2m_K", "total_precipitation_m",
                     "soil_moisture_vwc_layer1", "soil_ph_x10", "soil_organic_carbon_dg_kg", "sand_g_kg",
                     "clay_g_kg", "land_cover_code"]
missing_table = pd.DataFrame({"missing": df[important_columns].isna().sum(),
                              "percent": df[important_columns].isna().mean() * 100})
display(missing_table)

**3.3 Are the months complete and in order?** Every cell should have the same consecutive months. `groupby("cell_id").diff()` measures the gap between neighbouring rows **inside each cell only**, so a value from one cell can never leak into another cell.

*Expected output:* `0` gaps and every cell having the same number of months.

In [ ]:
# Sort the rows by cell and date, then check that each cell has complete, consecutive months.
# Later steps (12-month windows) assume this order, so we verify it here.
df = df.sort_values(["cell_id", "date"]).reset_index(drop=True)
df["month_number"] = df["date"].dt.year * 12 + df["date"].dt.month        # one integer per calendar month

# diff() subtracts the previous row; groupby makes it restart for every cell, so cells never mix.
month_gap = df.groupby("cell_id")["month_number"].diff()
print("Month gaps that are not exactly 1:", int((month_gap.dropna() != 1).sum()))

months_in_file = df["month_number"].nunique()
months_per_cell = df.groupby("cell_id")["month_number"].nunique()
print("Months in file:", months_in_file, "| cells with a different number of months:", int((months_per_cell != months_in_file).sum()))
print("Dates that are not the 1st of a month:", int((df["date"].dt.day != 1).sum()))

## 4. Data preprocessing and quality filtering

We create **new** columns and keep every raw column.

* `ndvi` = `ndvi_raw × 0.0001` (MODIS stores NDVI as an integer ×10,000). A safety check stops the notebook if the values already look like NDVI, so we can never scale twice.
* `temperature_c` = Kelvin − 273.15.
* `precipitation_mm` = metres × 1000.
* `month` (1 to 12) is used for seasonal comparisons.

In [ ]:
# Convert the raw columns into units that are easy to read, and validate the NDVI values.
# We multiply NDVI by 0.0001 exactly once; the safety check below stops us from scaling twice.
if df["ndvi_raw"].abs().max() <= 1.5:
    raise ValueError("ndvi_raw already looks like NDVI (-1 to 1). Do not multiply by 0.0001 again.")

df["ndvi"] = df["ndvi_raw"] * 0.0001                          # scale ONCE
df["temperature_c"] = df["temperature_2m_K"] - 273.15         # Kelvin -> Celsius
df["precipitation_mm"] = df["total_precipitation_m"] * 1000   # metres -> millimetres
df["month"] = df["date"].dt.month                             # 1..12

display(df[["ndvi", "temperature_c", "precipitation_mm"]].describe())
print("NDVI outside -1..1:", int(((df["ndvi"] < -1) | (df["ndvi"] > 1)).sum()))
print("ndvi_raw equal to the MODIS fill value -3000:", int((df["ndvi_raw"] == -3000).sum()))
print("Share of negative total_evaporation_m rows:", round((df["total_evaporation_m"] < 0).mean(), 3))
print("(Negative evaporation is normal for ERA5: it is a sign convention, not an error. We do not use evaporation here.)")

*Expected output:* NDVI roughly between −0.06 and 0.75, temperature about 10–40 °C, no NDVI outside −1..1, no fill values.

**NDVI quality flag.** `ndvi_summary_qa` is the MODIS quality flag (assumed MOD13A3 codes: 0 = good, 1 = marginal, 2 = snow/ice, 3 = cloudy). We look at it first, then apply one documented rule:

> **Quality rule:** an NDVI value is *reliable* only if the flag is 0 or 1. Other months stay in the table, but their NDVI is set to empty (`ndvi_valid`) so it is never used as if it were reliable. **No row is deleted and nothing is filled in.**

In [ ]:
# Mark which NDVI values are reliable, instead of deleting rows.
# ndvi_valid is NaN (empty) where the flag is not 0 or 1, so later averages skip those months automatically.
print("Quality flag counts:")
print(df["ndvi_summary_qa"].value_counts().sort_index())

df["ndvi_ok"] = df["ndvi_summary_qa"].isin([0, 1])      # True = reliable month
df["ndvi_valid"] = df["ndvi"].where(df["ndvi_ok"])      # where(): keep NDVI if True, otherwise NaN

print("\nShare of reliable observations:", round(df["ndvi_ok"].mean(), 4))
print("Cells with at least one unreliable month:", int((df.groupby("cell_id")["ndvi_ok"].min() == False).sum()))

*Expected output:* about 99.5% reliable observations and a few hundred cells with at least one unreliable month.

## 5. Feature engineering: one row per cell (`cell_features`)

Baghdad's vegetation is strongly seasonal, so we never compare a winter month with a summer month. Instead we use **full 12-month windows** (all seasons are included in both):

* **recent** = the latest 12 months in the file, **previous** = the 12 months before that.
* `recent_ndvi_change` = recent mean − previous mean.
* `seasonal_anomaly` = for each recent month, NDVI minus the same cell's average NDVI **in the same calendar month of earlier years**, averaged over the recent months.
* A mean is only trusted if at least `MIN_VALID_MONTHS` months were reliable. Otherwise it stays empty.

Only about 2–3 earlier years exist for each calendar month, so the seasonal comparison is a **modest, limited** baseline.

**5.1 Recent and previous 12-month windows.**

In [ ]:
# Summarise each cell over the latest 12 months ("recent") and the 12 months before ("previous").
# These windows give one number per cell that is not distorted by the summer/winter cycle.
last_date = df["date"].max()
last_month_number = last_date.year * 12 + last_date.month
df["months_ago"] = last_month_number - df["month_number"]     # 0 = the latest month in the file

recent_rows = df[df["months_ago"].between(0, 11)]             # between(): True if the value is inside the range
previous_rows = df[df["months_ago"].between(12, 23)]

# groupby + agg: one row per cell. Format: new_name=("column", "function").
recent_stats = recent_rows.groupby("cell_id").agg(
    recent_mean_ndvi=("ndvi_valid", "mean"),                  # mean() skips the empty (unreliable) months
    valid_months_recent=("ndvi_ok", "sum"),                   # True counts as 1, so sum = number of reliable months
    recent_temperature_c=("temperature_c", "mean"),
    recent_precipitation_mm=("precipitation_mm", "sum"),
    recent_soil_moisture=("soil_moisture_vwc_layer1", "mean"))
previous_stats = previous_rows.groupby("cell_id").agg(
    previous_mean_ndvi=("ndvi_valid", "mean"),
    valid_months_previous=("ndvi_ok", "sum"),
    previous_precipitation_mm=("precipitation_mm", "sum"))

# Not enough reliable months -> leave the average empty instead of inventing a value
recent_stats.loc[recent_stats["valid_months_recent"] < MIN_VALID_MONTHS, "recent_mean_ndvi"] = np.nan
previous_stats.loc[previous_stats["valid_months_previous"] < MIN_VALID_MONTHS, "previous_mean_ndvi"] = np.nan

print("Recent window:", recent_rows["date"].min().date(), "to", recent_rows["date"].max().date())
print("Previous window:", previous_rows["date"].min().date(), "to", previous_rows["date"].max().date())
print("Cells with too few reliable months (recent / previous):",
      int(recent_stats["recent_mean_ndvi"].isna().sum()), "/", int(previous_stats["previous_mean_ndvi"].isna().sum()))
print("Mean 12-month precipitation (mm), recent vs previous:", round(recent_stats["recent_precipitation_mm"].mean()), "vs",
      round(previous_stats["previous_precipitation_mm"].mean()))
print("Weather differs between years, so an NDVI change is not automatically land degradation.")

*Expected output:* the two window dates, no cell with too few reliable months, and different rainfall in the two years (a reminder that NDVI change is not the same as degradation).

**5.2 Seasonal anomaly.** Each recent month is compared with the same cell and the **same calendar month in earlier years only** (older than the recent window). At least 2 earlier observations are needed for each month.

In [ ]:
# Compare each recent month with the same cell's usual NDVI in the same calendar month of earlier years.
# This separates "it is simply summer" from "this cell is greener or browner than usual".
# Step 1: average NDVI of each cell for each calendar month, using only data older than the recent window.
baseline = (df[(df["months_ago"] >= 12) & df["ndvi_ok"]]
            .groupby(["cell_id", "month"])["ndvi"]
            .agg(baseline_ndvi="mean", baseline_obs="count")
            .reset_index())

# Step 2: attach that baseline to every reliable recent month (merge = join two tables on shared columns).
recent_reliable = df[(df["months_ago"] <= 11) & df["ndvi_ok"]].merge(baseline, on=["cell_id", "month"], how="left")
recent_reliable = recent_reliable[recent_reliable["baseline_obs"] >= 2]            # need at least 2 earlier years
recent_reliable["anomaly"] = recent_reliable["ndvi"] - recent_reliable["baseline_ndvi"]

# Step 3: average the monthly anomalies of each cell.
seasonal_stats = recent_reliable.groupby("cell_id").agg(seasonal_anomaly=("anomaly", "mean"),
                                                        seasonal_months_used=("anomaly", "count"))
seasonal_stats.loc[seasonal_stats["seasonal_months_used"] < MIN_VALID_MONTHS, "seasonal_anomaly"] = np.nan

print("Cells with a seasonal anomaly:", int(seasonal_stats["seasonal_anomaly"].notna().sum()))
print("Average number of earlier observations behind each monthly baseline:", round(recent_reliable["baseline_obs"].mean(), 2))

*Expected output:* almost all cells have a seasonal anomaly, built from about 2–3 earlier observations per month (a short baseline).

**5.3 Assemble `cell_features`.** We add the latest reliable NDVI and its date, the static soil attributes and the land-cover class.

* Land cover uses the ESA WorldCover codes. A single code at the cell centre cannot describe a whole 1 km cell, and class 50 (built-up) cannot separate housing from roads or industry.
* Cells that share exactly the same ERA5 temperature value belong to the same coarse (about 10 km) climate pixel. We record this (`climate_group_id`) only to check, later, whether the top candidates pile up in one climate pixel.

In [ ]:
# Build the one-row-per-cell table that every later step uses.
# Each earlier summary table has cell_id as its index, so join() lines the rows up by cell.
land_cover_names = {10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland", 50: "Built-up",
                    60: "Bare / sparse vegetation", 70: "Snow and ice", 80: "Permanent water",
                    90: "Herbaceous wetland", 95: "Mangroves", 100: "Moss and lichen"}

# Static information: the same value in every month of a cell, so "first" is enough (it skips empty values).
static_info = df.groupby("cell_id").agg(latitude=("latitude", "first"), longitude=("longitude", "first"),
                                        land_cover_code=("land_cover_code", "first"),
                                        soil_organic_carbon_dg_kg=("soil_organic_carbon_dg_kg", "first"),
                                        sand_g_kg=("sand_g_kg", "first"), clay_g_kg=("clay_g_kg", "first"),
                                        soil_ph_x10=("soil_ph_x10", "first"))

# Latest RELIABLE observation of each cell. tail(1) keeps the last row of each cell (rows are sorted by date).
latest_info = (df[df["ndvi_ok"]].groupby("cell_id").tail(1).set_index("cell_id")[["date", "ndvi"]]
               .rename(columns={"date": "latest_observation_date", "ndvi": "latest_ndvi"}))

cell_features = static_info.join([latest_info, recent_stats, previous_stats, seasonal_stats])
cell_features["recent_ndvi_change"] = cell_features["recent_mean_ndvi"] - cell_features["previous_mean_ndvi"]
cell_features["recent_valid_share"] = cell_features["valid_months_recent"] / 12
cell_features["land_cover_name"] = cell_features["land_cover_code"].map(land_cover_names)
cell_features["soil_missing"] = cell_features[["soil_organic_carbon_dg_kg", "sand_g_kg", "clay_g_kg"]].isna().any(axis=1)

# ngroup() gives the same number to all cells that share the same (rounded) temperature value.
cell_features["climate_group_id"] = cell_features.groupby(cell_features["recent_temperature_c"].round(6)).ngroup()
cell_features["cells_sharing_climate"] = cell_features.groupby("climate_group_id")["latitude"].transform("size")

print("Cells:", len(cell_features), "| cells with missing soil data (kept):", int(cell_features["soil_missing"].sum()))
print("Distinct climate values (coarse climate pixels):", cell_features["climate_group_id"].nunique())

**Verify the land-cover codes before using them.** We check that each cell has one code, that every code is in the WorldCover list, and that the mean NDVI per class makes sense.

*Expected output:* `True`, no unknown codes, and mean NDVI that is lowest for built-up/bare/grass and higher for tree cover and cropland.

In [ ]:
# Check that the land-cover codes mean what we think they mean.
# If the NDVI pattern per class looked wrong (for example built-up greener than trees), we should not trust the codes.
print("One land-cover code per cell:", bool(df.groupby("cell_id")["land_cover_code"].nunique().max() == 1))
print("Codes not in the WorldCover list:", sorted(set(cell_features["land_cover_code"].dropna()) - set(land_cover_names)) or "none")

land_cover_check = pd.DataFrame({"cells": cell_features["land_cover_name"].value_counts(),
                                 "mean_recent_ndvi": cell_features.groupby("land_cover_name")["recent_mean_ndvi"].mean()})
display(land_cover_check)

## 6. K-Means clustering (exploratory, unsupervised)

K-Means groups cells that behave similarly. **It does not predict planting success and it has no labels.**

We cluster only on three vegetation-history indicators, **not** on latitude/longitude, so the groups describe vegetation behaviour and not location:

`recent_mean_ndvi`, `recent_ndvi_change`, `seasonal_anomaly`

* `StandardScaler` puts the three indicators on the same scale; otherwise the indicator with the largest numbers would dominate.
* Cells missing any of the three indicators are **not clustered** (we do not invent values). They stay identifiable.
* We start with `N_CLUSTERS = 3` and do not tune it.

In [ ]:
# Cluster the cells by their vegetation history.
# Expected result: every cell with complete evidence gets a cluster_id (0, 1 or 2).
indicator_columns = ["recent_mean_ndvi", "recent_ndvi_change", "seasonal_anomaly"]    # also used by the screening rules

has_all_inputs = cell_features[indicator_columns].notna().all(axis=1)    # True for cells with all 3 indicators

# fit_transform learns the mean and spread of each indicator and rescales them (mean 0, spread 1).
scaled_inputs = StandardScaler().fit_transform(cell_features.loc[has_all_inputs, indicator_columns])

# n_init=10 runs K-Means 10 times from different starting points and keeps the best run.
kmeans = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=RANDOM_SEED)
cluster_ids = kmeans.fit_predict(scaled_inputs)                          # one cluster number per clustered cell

# "Int64" is a whole-number column that is allowed to hold empty values (cells that were not clustered).
cell_features["cluster_id"] = pd.Series(cluster_ids, index=cell_features.index[has_all_inputs]).astype("Int64")

print("Cells clustered:", int(has_all_inputs.sum()), "| not clustered (missing evidence):", int((~has_all_inputs).sum()))

### 6.1 Evaluating the clusters: the silhouette score

Because there are no planting labels, we **cannot** compute accuracy, precision, recall, F1 or ROC-AUC. The only evaluation available for K-Means is about the **cluster structure**.

| Question | Answer |
|---|---|
| **What is evaluated?** | How well separated the K-Means groups are in the three scaled indicators. |
| **Why this metric?** | It needs only the features and the cluster numbers, no labels. |
| **How is it obtained?** | For each cell: `s = (b − a) / max(a, b)`, where `a` = average distance to the other cells of its own cluster and `b` = average distance to the cells of the *nearest other* cluster. The score is the average of `s`. We use a random sample of cells (`SILHOUETTE_SAMPLE_SIZE`) to keep it fast. |
| **How to read it?** | From −1 to 1. Common rule of thumb (Kaufman & Rousseeuw): **above 0.5** reasonable structure, **0.25–0.5** weak to moderate, **below 0.25** little or no clear structure. Near 0 means groups overlap; negative means many cells look closer to another cluster. |
| **What does it tell us?** | Whether the groups are distinct or mostly cut through one continuous cloud of cells. |
| **What does it NOT tell us?** | Whether the groups matter in the real world, whether `k = 3` is the best choice (we did not tune it), whether any group is degraded, or anything about planting success. |

In [ ]:
# Compute the silhouette score of the K-Means clustering and explain it in words.
# Expected result: one number between -1 and 1 plus a plain-language reading.
silhouette = silhouette_score(scaled_inputs, cluster_ids, sample_size=SILHOUETTE_SAMPLE_SIZE, random_state=RANDOM_SEED)
print(f"Silhouette score (sample of {SILHOUETTE_SAMPLE_SIZE} cells): {silhouette:.3f}")

if silhouette < 0.25:
    silhouette_reading = "little or no clear structure: the clusters overlap and are only a rough grouping"
elif silhouette < 0.5:
    silhouette_reading = "weak to moderate structure: the clusters are partly separated but overlap"
else:
    silhouette_reading = "reasonable or strong structure"
print("Reading:", silhouette_reading)
print("Remember: this measures separation between groups only. It says nothing about degradation or planting success.")

### 6.2 Interpreting the clusters

**Interpret the clusters from their measured statistics, not from their ID numbers** (the numbers 0, 1, 2 are arbitrary). The table shows the mean and median of each indicator per cluster. We give each cluster a descriptive name from its rank by recent NDVI. We do **not** call any cluster "degraded".

*What to look for:* clusters often differ mainly in NDVI **level**. A "declining" cluster exists only if its mean change is clearly negative.

In [ ]:
# Describe each cluster with simple statistics and give it a readable name.
# Expected result: a table with one row per cluster and a bar chart of the three indicators.
cluster_summary = cell_features.groupby("cluster_id")[indicator_columns].agg(["mean", "median"])
cluster_summary.columns = ["_".join(column) for column in cluster_summary.columns]    # flatten ("a","mean") -> "a_mean"
cluster_summary["cells"] = cell_features.groupby("cluster_id").size()

cell_features["is_built_up"] = cell_features["land_cover_code"] == 50           # True/False; the mean of True/False is a share
cluster_summary["share_built_up"] = cell_features.groupby("cluster_id")["is_built_up"].mean()
display(cluster_summary)

# Name each cluster by its rank of recent NDVI (rank 1 = lowest). The mean change is shown in the name.
ndvi_rank = cluster_summary["recent_mean_ndvi_mean"].rank().astype(int)
rank_words = {1: "lowest", 2: "middle", 3: "highest"}
cluster_names = {}
for cluster in cluster_summary.index:
    rank_word = rank_words[ndvi_rank[cluster]]
    mean_change = cluster_summary.loc[cluster, "recent_ndvi_change_mean"]
    cluster_names[cluster] = f"Cluster {cluster}: {rank_word} recent NDVI, mean change {mean_change:+.2f}"
cell_features["cluster_label"] = cell_features["cluster_id"].map(cluster_names).fillna("Not clustered (insufficient evidence)")

# One small bar chart per indicator: the mean value of each cluster.
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for axis, indicator in zip(axes, indicator_columns):
    axis.bar([cluster_names[c].split(":")[0] for c in cluster_summary.index], cluster_summary[indicator + "_mean"], color="#4c78a8")
    axis.set_title("Mean " + indicator)
    axis.set_xlabel("Cluster")
    axis.set_ylabel("Mean value")
    axis.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

*Expected output:* three clusters: one with low NDVI (many built-up cells), one with high NDVI, and one smaller cluster whose mean change and seasonal anomaly are clearly negative.

## 7. Rule-based screening: where might intervention be needed? (separate from K-Means)

Three explainable signals per cell. **All thresholds are in the Settings cell (Section 1).** Each one is an **MVP assumption**, not an established ecological limit.

| Signal | Meaning |
|---|---|
| `flag_low_ndvi` | recent 12-month NDVI is in the lowest part of all Baghdad cells (relative to the city, so a fixed share of cells is always flagged) |
| `flag_declining` | recent NDVI is lower than the previous 12 months by at least `DECLINE_THRESHOLD` |
| `flag_below_seasonal` | recent months are below the cell's usual level for the same calendar months by at least `SEASONAL_DROP_THRESHOLD` |

In [ ]:
# Turn the three NDVI indicators into True/False signals using the thresholds from Section 1.
# Expected result: counts of flagged cells per signal and a simple trend label per cell.
# For reference: the two change thresholds are about one standard deviation of the city-wide values.
print("Std of recent_ndvi_change:", round(cell_features["recent_ndvi_change"].std(), 3),
      "| std of seasonal_anomaly:", round(cell_features["seasonal_anomaly"].std(), 3))

low_ndvi_cutoff = cell_features["recent_mean_ndvi"].quantile(LOW_NDVI_PERCENTILE / 100)   # quantile(0.25) = value below which 25% of cells fall
cell_features["flag_low_ndvi"] = cell_features["recent_mean_ndvi"] <= low_ndvi_cutoff            # empty values give False
cell_features["flag_declining"] = cell_features["recent_ndvi_change"] <= DECLINE_THRESHOLD
cell_features["flag_below_seasonal"] = cell_features["seasonal_anomaly"] <= SEASONAL_DROP_THRESHOLD

# np.select(conditions, labels): the first condition that is True decides the label; "default" is used if none is True.
cell_features["vegetation_trend"] = np.select(
    [cell_features["recent_ndvi_change"].isna(), cell_features["recent_ndvi_change"] <= DECLINE_THRESHOLD,
     cell_features["recent_ndvi_change"] >= -DECLINE_THRESHOLD],
    ["Unknown", "Declining", "Increasing"], default="Stable")

print(f"Low-NDVI cutoff (recent 12-month mean NDVI, {LOW_NDVI_PERCENTILE}th percentile): {low_ndvi_cutoff:.3f}")
print("Flagged cells -> low NDVI:", int(cell_features["flag_low_ndvi"].sum()),
      "| declining:", int(cell_features["flag_declining"].sum()),
      "| below seasonal:", int(cell_features["flag_below_seasonal"].sum()))
print(cell_features["vegetation_trend"].value_counts().to_dict())

**Intervention screening category.** A single signal is never enough for the top category:

| Condition | `intervention_category` |
|---|---|
| fewer than `MIN_USABLE_INDICATORS` indicators available | Insufficient evidence |
| 2 or more signals **and at least one is a change over time** (decline or below-seasonal) | Higher screening priority |
| 1 or more signals | Moderate screening priority |
| no signals | Lower screening priority |

So **low NDVI alone can reach "Moderate" at most**. These are screening categories, not ecological diagnoses.

In [ ]:
# Combine the signals into one screening category per cell.
# Expected result: counts of cells in each category (most cells should be "Lower", a minority "Higher").
flag_columns = ["flag_low_ndvi", "flag_declining", "flag_below_seasonal"]
usable_indicators = cell_features[indicator_columns].notna().sum(axis=1)        # how many of the 3 indicators exist for each cell

cell_features["need_signals"] = cell_features[flag_columns].sum(axis=1)        # True counts as 1, so this is 0 to 3
cell_features["temporal_signals"] = cell_features["flag_declining"].astype(int) + cell_features["flag_below_seasonal"].astype(int)

need_conditions = [usable_indicators < MIN_USABLE_INDICATORS,
                   (cell_features["need_signals"] >= 2) & (cell_features["temporal_signals"] >= 1),
                   cell_features["need_signals"] >= 1]
need_labels = ["Insufficient evidence", "Higher screening priority", "Moderate screening priority"]
cell_features["intervention_category"] = np.select(need_conditions, need_labels, default="Lower screening priority")

print(cell_features["intervention_category"].value_counts())

## 8. Rule-based screening: preliminary greening potential (separate from need)

This is a transparent, preliminary look at the **soil**. It is not a trained model and not a planting-success prediction.

* We use only **local** information: soil organic carbon and sand share (a simple texture indicator), from a 250 m soil map.
* We do **not** use soil pH (it barely varies across Baghdad), clay (no justified limit), or temperature / rainfall / soil moisture: the climate data are about 10 km wide, so neighbouring cells share the same value and it would give a false picture of local differences. Climate is only shown as context.
* There is no universal ideal soil, so the limits are **relative to Baghdad** and are MVP assumptions: *organic carbon at or above the city median* and *sand share at or below the city's `SAND_UPPER_QUANTILE`* each count as one favourable signal.

| Condition | `greening_potential_category` |
|---|---|
| water cell, or any soil value missing | Needs further assessment (missing soil is **not** treated as poor) |
| 2 favourable soil signals | More promising |
| 1 favourable soil signal | Mixed |
| 0 favourable soil signals | Less promising |

Built-up cells are **not** excluded or approved here; they are flagged for land-use inspection in Section 9.

In [ ]:
# Rate each cell's soil relative to the rest of Baghdad.
# Expected result: counts per potential category; cells without soil data are kept as "Needs further assessment".
soc_median = cell_features["soil_organic_carbon_dg_kg"].median()                       # MVP assumption: "richer than the typical Baghdad cell"
sand_upper_limit = cell_features["sand_g_kg"].quantile(SAND_UPPER_QUANTILE)            # MVP assumption: "not among the sandiest cells"

good_carbon = cell_features["soil_organic_carbon_dg_kg"] >= soc_median
good_texture = cell_features["sand_g_kg"] <= sand_upper_limit
favourable_signals = good_carbon.astype(int) + good_texture.astype(int)               # 0, 1 or 2

# The order matters: water and missing soil are checked first, so they can never be called "poor".
potential_conditions = [cell_features["land_cover_code"] == 80,                       # water: mixed cell
                        cell_features["soil_missing"],                                # unknown, NOT "poor"
                        favourable_signals == 2,
                        favourable_signals == 1]
potential_labels = ["Needs further assessment", "Needs further assessment", "More promising", "Mixed"]
cell_features["greening_potential_category"] = np.select(potential_conditions, potential_labels, default="Less promising")

print(f"Organic carbon median: {soc_median / 10:.1f} g/kg | sand upper limit: {sand_upper_limit:.0f} g/kg")
print(cell_features["greening_potential_category"].value_counts())
print("Soil pH range across cells:", cell_features["soil_ph_x10"].min() / 10, "to", cell_features["soil_ph_x10"].max() / 10,
      "(too narrow to separate cells, so not used)")

## 9. Candidate ranking and selection

**Ranking order:** (1) evidence of need, (2) greening potential only as a *tie-breaker inside the same need level*, (3) data confidence.

First every cell is placed in a **group**, so these cases stay distinguishable:

| `candidate_group` | Meaning |
|---|---|
| Vegetated area with decline | not in the lowest-NDVI group, but declining or below its usual seasonal level (maintenance or restoration may be relevant) |
| Low-vegetation open area | low recent NDVI and not built-up or water: deserves a further greening assessment |
| Built-up cell (land use must be inspected) | a 1 km cell is too coarse to say where a planting location could be |
| Water / mixed cell (inspect) | mixed cell: check what the land really is |
| No clear signal / Insufficient evidence | nothing to act on |

`final_priority` comes from the need level only: **High** = Higher screening priority, **Medium** = Moderate, **Low** = Lower. **Built-up and water cells are capped at Medium**, because we cannot see an actual planting location. Potential and confidence then order the cells inside each level.

In [ ]:
# Assign a candidate group, a final priority, a data-confidence level and a rank to every cell.
# Expected result: a table counting cells by group and priority (built-up and water never reach "High").
is_built_up = cell_features["land_cover_code"] == 50
is_water = cell_features["land_cover_code"] == 80
is_insufficient = cell_features["intervention_category"] == "Insufficient evidence"

group_conditions = [is_insufficient, is_built_up, is_water, cell_features["flag_low_ndvi"], cell_features["temporal_signals"] >= 1]
group_labels = ["Insufficient evidence", "Built-up cell (land use must be inspected)", "Water / mixed cell (inspect)",
                "Low-vegetation open area", "Vegetated area with decline"]
cell_features["candidate_group"] = np.select(group_conditions, group_labels, default="No clear signal")

# Convert the need category into points (2, 1, 0). Cells with insufficient evidence get NaN (no points).
need_points = cell_features["intervention_category"].map({"Higher screening priority": 2, "Moderate screening priority": 1,
                                                          "Lower screening priority": 0})
# clip(upper=1) caps the points at 1 for built-up and water cells, so they can reach "Medium" at most.
effective_points = need_points.where(~(is_built_up | is_water), need_points.clip(upper=1))
cell_features["final_priority"] = np.select([is_insufficient, effective_points == 2, effective_points == 1],
                                            ["Insufficient evidence", "High", "Medium"], default="Low")

# Data confidence from how many of the last 12 months have reliable NDVI
cell_features["data_confidence"] = np.select([cell_features["valid_months_recent"] >= 11, cell_features["valid_months_recent"] >= MIN_VALID_MONTHS],
                                             ["High", "Medium"], default="Low")

# Sort order: need first, then number of signals, then potential, then confidence, then size of the decline.
ranking_table = pd.DataFrame({
    "effective_points": effective_points,
    "need_signals": cell_features["need_signals"],
    "potential_order": cell_features["greening_potential_category"].map({"More promising": 3, "Mixed": 2, "Less promising": 1, "Needs further assessment": 0}),
    "confidence_order": cell_features["data_confidence"].map({"High": 2, "Medium": 1, "Low": 0}),
    "recent_ndvi_change": cell_features["recent_ndvi_change"]})
ranked_ids = ranking_table.sort_values(["effective_points", "need_signals", "potential_order", "confidence_order", "recent_ndvi_change"],
                                       ascending=[False, False, False, False, True], na_position="last").index
cell_features["rank"] = pd.Series(np.arange(1, len(ranked_ids) + 1), index=ranked_ids)      # rank 1 = first in the sorted order

display(pd.crosstab(cell_features["candidate_group"], cell_features["final_priority"]))     # crosstab counts cells per pair of categories

**Readable reasons and uncertainty.** `short_reason` quotes the actual indicator values. `data_uncertainty` lists what is missing or limited. Missing soil is written as *missing*, never as *suitable* or *poor*, and climate is not described as a local difference.

In [ ]:
# Write a human-readable reason and an uncertainty note for every cell.
# Expected result: two text columns; the first 3 rows are printed as an example.
def as_text(series, number_format="{:.2f}"):
    # A function is a reusable block of code: here it turns numbers into text and writes "n/a" for empty values.
    text = series.map(number_format.format)
    return text.where(series.notna(), "n/a")

# Reason: which signals fired
signals_text = (cell_features["flag_low_ndvi"].map({True: "low recent NDVI; ", False: ""})
                + cell_features["flag_declining"].map({True: "NDVI fell versus the previous 12 months; ", False: ""})
                + cell_features["flag_below_seasonal"].map({True: "below its usual level for the same months; ", False: ""}))
signals_text = signals_text.str.rstrip("; ").replace("", "no signal")

# Reason: soil values, or a clear "missing" message
soil_text = ("organic carbon " + as_text(cell_features["soil_organic_carbon_dg_kg"] / 10, "{:.1f}") + " g/kg, sand "
             + as_text(cell_features["sand_g_kg"], "{:.0f}") + " g/kg")
soil_text = soil_text.mask(cell_features["soil_missing"], "soil data missing (not treated as poor)")     # mask(): replace where True

cell_features["short_reason"] = ("Recent NDVI " + as_text(cell_features["recent_mean_ndvi"])
                                 + "; change vs previous 12 months " + as_text(cell_features["recent_ndvi_change"], "{:+.2f}")
                                 + "; vs usual same-month level " + as_text(cell_features["seasonal_anomaly"], "{:+.2f}")
                                 + ". Signals: " + signals_text + ". Soil: " + soil_text
                                 + ". Group: " + cell_features["candidate_group"] + ".")

# Uncertainty: what is missing or limited for this cell
land_cover_notes = {10: "already tree cover (care may matter more than new planting); ",
                    40: "cropland (check land use before any greening); ",
                    50: "1 km cell centred on built-up land: cannot tell whether open space exists; ",
                    60: "centre is bare/sparse land (not automatically plantable); ",
                    80: "mixed water cell; "}
soil_note = cell_features["soil_missing"].map({True: "soil data missing; ", False: ""})
reliability_note = (cell_features["valid_months_recent"] < 12).map({True: "some recent months have unreliable NDVI; ", False: ""})
season_note = cell_features["seasonal_anomaly"].isna().map({True: "no seasonal comparison available; ", False: ""})
date_note = (cell_features["latest_observation_date"] < last_date).map({True: "latest reliable NDVI is older than the end of the data; ", False: ""})
cover_note = cell_features["land_cover_code"].map(land_cover_notes).fillna("")

cell_features["data_uncertainty"] = (soil_note + reliability_note + season_note + date_note + cover_note
                                     + "land cover is a single value at the cell centre; ").str.rstrip("; ")

print(cell_features[["short_reason", "data_uncertainty"]].head(3).to_string())

**Top candidates.** The table shows the 20 best-ranked cells. This ordering is **preliminary and not validated against real planting outcomes**: these are candidates for further assessment only.

In [ ]:
# Show the best-ranked cells and what kind of cells they are.
# Expected result: a top-20 table and a short description of the top 100 (groups and land cover).
ranked_cells = cell_features.sort_values("rank")
top_100 = ranked_cells.head(TOP_N_CANDIDATES)

print("Top-100 candidate groups:", top_100["candidate_group"].value_counts().to_dict())
print("Top-100 land cover:", top_100["land_cover_name"].value_counts().to_dict())

print("\nTOP 20 (preliminary ordering, NOT validated against real planting outcomes; candidates for further assessment only):")
display(ranked_cells.head(20).reset_index()[["rank", "cell_id", "latitude", "longitude", "recent_mean_ndvi", "recent_ndvi_change",
                                            "intervention_category", "greening_potential_category", "final_priority",
                                            "candidate_group", "land_cover_name"]])

## 10. Visualizations and results interpretation

Four plots (the K-Means cluster chart was already drawn in Section 6). There is no interactive map yet.

1. Monthly mean NDVI: the seasonal cycle of the whole city (reliable months only).
2. and 3. How many cells fall in each screening category and each soil-potential category.
4. A map-style scatter of all cells coloured by final priority.

In [ ]:
# Draw the four summary plots.
# Expected result: a clear yearly NDVI cycle, two bar charts, and a map-like scatter coloured by priority.

# 1) Mean NDVI over time (reliable months only)
plt.figure(figsize=(8, 3.5))
df.groupby("date")["ndvi_valid"].mean().plot(marker="o")
plt.title("Monthly mean NDVI across Baghdad cells (reliable months only)")
plt.xlabel("Month")
plt.ylabel("NDVI")
plt.grid(alpha=0.3)
plt.show()

# 2) and 3) category counts side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
cell_features["intervention_category"].value_counts().plot.barh(ax=axes[0], color="#7a8f84")
axes[0].set_title("Intervention screening categories")
axes[0].set_xlabel("Number of cells")
axes[0].set_ylabel("Category")
cell_features["greening_potential_category"].value_counts().plot.barh(ax=axes[1], color="#2e9e5b")
axes[1].set_title("Preliminary greening potential (soil-based)")
axes[1].set_xlabel("Number of cells")
axes[1].set_ylabel("Category")
plt.tight_layout()
plt.show()

# 4) Map-style scatter of all cells coloured by final priority (one scatter call per colour gives a legend)
priority_colors = {"Insufficient evidence": "#bbbbbb", "Low": "#9ecae1", "Medium": "orange", "High": "red"}
plt.figure(figsize=(7, 6))
for priority_name, color in priority_colors.items():
    group = cell_features[cell_features["final_priority"] == priority_name]
    plt.scatter(group["longitude"], group["latitude"], s=6, color=color, label=f"{priority_name} ({len(group)})")
plt.title("Baghdad cells by final priority (preliminary)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend(loc="lower left", fontsize=8)
plt.show()

*How to read them.* The first plot shows the normal seasonal swing (this is why we never compare different months directly). The two bar charts show that most cells have no strong signal and only a minority are screened as higher priority. The map shows **where** the priorities are, but a coloured dot is a cell to check on satellite imagery or in the field, not a planting site.

## 11. Exporting results

Four files are written to Colab's working directory (the *Files* panel on the left). Nothing is downloaded automatically.

| File | Content |
|---|---|
| `baghdad_cell_screening.csv` | every cell, sorted by rank |
| `baghdad_top_candidates.csv` | top candidates (High or Medium priority) |
| `baghdad_screening_summary.csv` | number of cells in each category |
| `baghdad_scoring_metadata.json` | definitions, thresholds and limitations |

In [ ]:
# Build the output tables and save them as CSV files.
# Expected result: 6011 rows in the full table and TOP_N_CANDIDATES rows in the top-candidates table.
output_columns = ["cell_id", "latitude", "longitude", "latest_observation_date", "latest_ndvi", "recent_mean_ndvi",
                  "recent_ndvi_change", "vegetation_trend", "seasonal_anomaly", "recent_valid_share", "cluster_label", "intervention_category",
                  "need_signals", "greening_potential_category", "candidate_group", "final_priority", "rank", "data_confidence",
                  "land_cover_name", "soil_missing", "cells_sharing_climate", "short_reason", "data_uncertainty"]

all_cells_table = cell_features.reset_index().sort_values("rank")[output_columns]
top_candidates = all_cells_table[all_cells_table["final_priority"].isin(["High", "Medium"])].head(TOP_N_CANDIDATES)

# Summary: number of cells in every category (one value_counts table per variable, stacked with concat).
summary_variables = ["cluster_label", "intervention_category", "greening_potential_category", "candidate_group", "final_priority"]
counts_by_variable = {}
for column in summary_variables:
    counts_by_variable[column] = cell_features[column].value_counts()
summary_table = pd.concat(counts_by_variable, names=["variable", "category"]).rename("number_of_cells").reset_index()

all_cells_table.to_csv("baghdad_cell_screening.csv", index=False)
top_candidates.to_csv("baghdad_top_candidates.csv", index=False)
summary_table.to_csv("baghdad_screening_summary.csv", index=False)

In [ ]:
# Save the definitions, thresholds and limitations next to the data, so the results can be explained later.
# JSON is a plain-text format that stores nested dictionaries; open it in any text editor.
metadata = {
    "project": "GreenSite AI - Baghdad greening MVP",
    "dataset_first_date": str(df["date"].min().date()), "dataset_last_date": str(df["date"].max().date()),
    "cells": int(len(cell_features)),
    "ndvi": "ndvi = ndvi_raw * 0.0001 (applied once); reliable only if ndvi_summary_qa is 0 or 1",
    "windows": f"recent = latest 12 months ({recent_rows['date'].min().date()} to {recent_rows['date'].max().date()}); previous = the 12 months before",
    "feature_definitions": {"recent_mean_ndvi": "mean reliable NDVI in the recent window", "recent_ndvi_change": "recent mean minus previous mean",
                            "seasonal_anomaly": "mean of (NDVI - same cell's mean NDVI of the same calendar month in earlier years) over recent months",
                            "min_valid_months": MIN_VALID_MONTHS},
    "kmeans": {"features": indicator_columns, "n_clusters": N_CLUSTERS, "n_init": 10, "random_state": RANDOM_SEED, "scaler": "StandardScaler",
               "silhouette_sample_3000": float(silhouette), "cells_not_clustered": int((~has_all_inputs).sum()),
               "note": "Exploratory unsupervised grouping; NOT used for the final priority; not a label of degradation."},
    "screening_thresholds_mvp_assumptions": {"low_ndvi_percentile": LOW_NDVI_PERCENTILE, "low_ndvi_cutoff": float(low_ndvi_cutoff),
                                             "decline_threshold": DECLINE_THRESHOLD, "seasonal_drop_threshold": SEASONAL_DROP_THRESHOLD,
                                             "min_usable_indicators": MIN_USABLE_INDICATORS},
    "greening_potential_rules": {"soil_organic_carbon_median_dg_kg": float(soc_median), "sand_upper_limit_g_kg": float(sand_upper_limit),
                                 "rule": "water or any missing soil value -> Needs further assessment; 2 favourable signals -> More promising; 1 -> Mixed; 0 -> Less promising",
                                 "not_used": "soil pH (almost no variation), clay (no justified limit), temperature/precipitation/soil moisture (coarse climate, shared by many cells)"},
    "category_definitions": {"Higher screening priority": ">=2 signals including >=1 change over time",
                             "Moderate screening priority": ">=1 signal", "Lower screening priority": "no signal",
                             "Insufficient evidence": "fewer than 2 usable indicators",
                             "final_priority": "High/Medium/Low from the need level only; built-up and water cells are capped at Medium; potential and confidence only order cells inside a level"},
    "limitations": [
        "No labels of planting success or failure exist: nothing here predicts planting success or probability.",
        "Low NDVI does not prove degradation; a promising soil rating does not mean a site can be planted.",
        "Only about 3.4 years of data: the seasonal baseline uses 2-3 earlier observations per calendar month.",
        "Year-to-year NDVI change is affected by weather, crop cycles and land use; the indicators cannot separate drought from other causes.",
        "Climate (ERA5-Land, NASA POWER) is coarse and shared by many neighbouring cells, so it is not used to separate cells.",
        "Soil and land cover are static, sampled at the cell centre; a 1 km cell is mixed and class 50 cannot separate housing, roads or industry.",
        "About 12% of cells have no soil data; they are kept and marked, not treated as poor.",
        "All thresholds are MVP assumptions, not ecological standards. The data was not fully cross-checked against every original source."],
}
with open("baghdad_scoring_metadata.json", "w", encoding="utf-8") as json_file:
    json.dump(metadata, json_file, indent=2)

print("Saved 4 files. Rows:", len(all_cells_table), "(all cells) /", len(top_candidates), "(top candidates)")

## 12. Evaluation and verification

GreenSite AI has **no ground truth** (no record of where planting worked), so the usual supervised scores (accuracy, precision, recall, F1, ROC-AUC) **do not exist here and are not invented**. Instead we run five different kinds of checks. Each answers a different question, and none of them proves that a site is good for planting.

| Kind of check | Question it answers | Where |
|---|---|---|
| **Data-quality validation** | Is the data complete, in order, correctly scaled and unchanged? | 3, 4, 12.1 |
| **Clustering evaluation** | Are the K-Means groups well separated? | 6.1 |
| **Rule-based screening checks** | Do the rules behave sensibly and agree with a second, independent method? | 12.2 |
| **Geographic validation** | Are the top candidates spread out, or piled up in one place? | 12.3 |
| **Real-world outcome validation** | Did planting actually succeed in the selected cells? | **Not done: no data** |

**12.1 Data-quality and promise checks.** Each line must be `True`. They check that the raw data was not changed, that NDVI was scaled once, that histories never mix between cells, that unreliable NDVI was never used, and that the notebook made none of the forbidden shortcuts (coordinates inside K-Means, a fake planting-success column, dropping cells with missing soil).

*What this tells us:* the pipeline did what it promises. *What it does NOT tell us:* that the data itself is correct (we cannot see the original satellite files here).

In [ ]:
# Verify the promises this notebook makes. Every check should be True.
# The raw-data check uses the fingerprint saved in Section 2.
checks = {
    "Raw data unchanged": pd.util.hash_pandas_object(df_raw, index=False).sum() == raw_fingerprint,
    "ndvi_raw values unchanged in the working copy": df["ndvi_raw"].sum() == df_raw["ndvi_raw"].sum(),
    "NDVI scaled once (inside -1..1 and equal to ndvi_raw x 0.0001)": bool(df["ndvi"].abs().max() <= 1 and np.allclose(df["ndvi"], df["ndvi_raw"] * 0.0001)),
    "Histories stay inside each cell (no month gaps)": bool((df.groupby("cell_id")["month_number"].diff().dropna() == 1).all()),
    "Unreliable NDVI never used (ndvi_valid empty where flag is not 0/1)": bool(df.loc[~df["ndvi_ok"], "ndvi_valid"].isna().all()),
    "K-Means inputs exclude coordinates": not {"latitude", "longitude"} & set(indicator_columns),
    "No planting-success column created": not any("suitab" in column.lower() or "success" in column.lower() for column in all_cells_table.columns),
    "Every cell is in the final table (none dropped, even with missing soil)": len(all_cells_table) == df["cell_id"].nunique(),
    "Coordinates valid in exported file": bool(all_cells_table["latitude"].between(-90, 90).all() and all_cells_table["longitude"].between(-180, 180).all()),
}
display(pd.Series(checks, name="passed"))
print("All checks passed:", all(checks.values()))
print("Cells with missing soil kept:", int(cell_features["soil_missing"].sum()), "of", len(cell_features), "cells")

for file_name in ["baghdad_cell_screening.csv", "baghdad_top_candidates.csv", "baghdad_screening_summary.csv", "baghdad_scoring_metadata.json"]:
    print(f"{file_name:35s} {os.path.getsize(file_name) / 1024:8.1f} KB")

**12.2 Do the rules and K-Means agree?** The screening rules and K-Means are two *independent* ways to look at the same NDVI history. If a K-Means cluster has a clearly negative mean change, we expect it to contain a high share of cells that the rules flag as declining.

* *What this tells us:* the two methods tell a consistent story.
* *What it does NOT tell us:* that either is right. Two methods can agree and still both be wrong (for example if a crop cycle changed, not the land).

In [ ]:
# Compare K-Means groups with the screening flags: what share of each cluster is flagged by each rule?
# Expected result: the cluster with the clearly negative mean change should have the highest share of declining / below-seasonal flags.
# The mean of a True/False column is the share of True, so this table shows shares between 0 and 1.
display(cell_features.groupby("cluster_label")[flag_columns].mean())

**12.3 Geographic check: are the top candidates piled up in one place?** We count how many of the top 100 candidates share the same coarse climate pixel and compare it with that pixel's share of all cells.

* *What this tells us:* whether the ranking is concentrated in one region. A ratio far above 1 means the top candidates cluster geographically.
* *What it does NOT tell us:* the cause. NDVI is measured locally, so clustering usually means a **regional pattern** (crop cycles, irrigation, land-use change) rather than coarse climate driving the score, but this notebook cannot separate those explanations. Check such areas on satellite images first.

In [ ]:
# Check whether the top 100 candidates concentrate in a few coarse climate pixels.
# Expected result: a small table and one sentence with the concentration ratio.
concentration = pd.DataFrame({"share_of_top_100": top_100["climate_group_id"].value_counts(normalize=True),
                              "share_of_all_cells": cell_features["climate_group_id"].value_counts(normalize=True)}).dropna()
concentration = concentration.sort_values("share_of_top_100", ascending=False).head(5)

print("Climate pixels with the most top-100 candidates (compared with their share of all cells):")
display(concentration)
biggest_ratio = concentration["share_of_top_100"].iloc[0] / concentration["share_of_all_cells"].iloc[0]
print(f"Largest climate pixel holds {concentration['share_of_top_100'].iloc[0]:.0%} of the top 100 vs {concentration['share_of_all_cells'].iloc[0]:.0%} of all cells (x{biggest_ratio:.1f}).")
print("Interpretation: a regional pattern is likely (crop cycles, irrigation, land-use change). Verify these areas on satellite images before trusting them.")

**12.4 Evaluation summary.** One block per component, with the evidence taken from the results above.

In [ ]:
# Print a plain-language evaluation summary: what was checked, the evidence, and what it does and does not tell us.
# Expected result: six short blocks that can be read aloud to a judge or teammate.
need_counts = cell_features["intervention_category"].value_counts().to_dict()
potential_counts = cell_features["greening_potential_category"].value_counts().to_dict()

evaluation_summary = [
    {"part": "Data-quality validation",
     "evidence": f"all promise checks passed: {all(checks.values())}; reliable NDVI share: {df['ndvi_ok'].mean():.3f}",
     "tells": "the pipeline kept the raw data intact, scaled NDVI once and never mixed cells",
     "not": "that the source data is correct or complete in the real world"},
    {"part": "K-Means clustering (silhouette)",
     "evidence": f"silhouette = {silhouette:.3f} -> {silhouette_reading}",
     "tells": "how well separated the groups are in the three NDVI indicators",
     "not": "whether k = 3 is best (not tuned), whether groups are degraded, or anything about planting success"},
    {"part": "Rule-based need screening",
     "evidence": f"category counts: {need_counts}",
     "tells": "how many cells show NDVI warning signals, and that the rules do not label everything or nothing",
     "not": "the cause of a signal (drought, crop cycle, land-use change) or whether intervention is needed"},
    {"part": "Rule-based greening potential",
     "evidence": f"category counts: {potential_counts}",
     "tells": "which cells have relatively richer / less sandy soil than the Baghdad median",
     "not": "that a site can be planted (no water, salinity, land-ownership or land-use information)"},
    {"part": "Geographic validation",
     "evidence": f"largest climate pixel holds {concentration['share_of_top_100'].iloc[0]:.0%} of the top 100 (x{biggest_ratio:.1f} its share of cells)",
     "tells": "whether the ranking is concentrated in one region",
     "not": "why it is concentrated"},
    {"part": "Real-world outcome validation",
     "evidence": "NOT DONE: there are no records of planting success or failure",
     "tells": "nothing yet",
     "not": "success rates, accuracy or probabilities: no such claim is supported by this notebook"},
]
for item in evaluation_summary:
    print(item["part"].upper())
    print("  Evidence:", item["evidence"])
    print("  Tells us:", item["tells"])
    print("  Does NOT tell us:", item["not"])
    print()

## 13. Final summary and limitations

In [ ]:
# Print the headline results in one place.
# Expected result: a short summary that can be copied into a presentation slide.
print("Cells analysed:", len(cell_features), "| data from", df["date"].min().date(), "to", df["date"].max().date())
print("K-Means clusters:", N_CLUSTERS, "| silhouette:", round(silhouette, 3), "(", silhouette_reading, ")")
print("Final priority counts:", cell_features["final_priority"].value_counts().to_dict())
print("Top candidate file:", "baghdad_top_candidates.csv", "with", len(top_candidates), "cells")

### What these results mean

* The ranking tells us **which cells to inspect first**. It does not tell us where trees will grow.
* K-Means gives an exploratory picture of vegetation behaviour. It is **not** used in the ranking.
* The rules use simple, adjustable thresholds that are **assumptions**, not ecological standards. Change them in Section 1 and re-run to see how sensitive the ranking is.

### Main limitations

1. **No labels.** There is no record of planting success or failure, so nothing is predicted and no accuracy exists.
2. **Low NDVI is not degradation.** It can also mean desert, built-up land, bare soil or a normal crop cycle.
3. **Short history.** About 3.4 years of data; the seasonal baseline uses only 2–3 earlier observations per calendar month.
4. **Weather and land use mix into NDVI change.** The indicators cannot separate drought from crop rotation or construction.
5. **Coarse climate.** ERA5/POWER values are shared by many neighbouring cells, so climate is not used to separate cells.
6. **Static, point-sampled soil and land cover.** A 1 km cell is mixed, and "built-up" cannot be split into housing, roads or industry.
7. **Missing soil (about 12% of cells)** is kept and marked, never treated as poor.
8. **Top candidates may be geographically concentrated** (see 12.3) and must be checked on satellite images.

### Natural next step

Check the top candidates on satellite imagery and in the field. Records of real planting outcomes would be the first step toward a genuine supervised model.

### Optional: download the files

Run this last cell only when you want the files on your computer (your browser may ask to allow multiple downloads). You can also download them from the *Files* panel on the left.

In [ ]:
# Download the four result files to your computer.
#files.download("baghdad_cell_screening.csv")
#files.download("baghdad_top_candidates.csv")
#files.download("baghdad_screening_summary.csv")
#files.download("baghdad_scoring_metadata.json")

In [ ]:
# ==============================================================================
# Step 10A — Map data: land-cover screening + spaced Top-20 shortlist (distance filter)
# Run this AFTER the main notebook (it needs `cell_features`).
# It does NOT change K-Means, the screening rules, the ranks or the priorities.
# Expected result: a few printed counts and 4 CSV files for the map / Streamlit.
# ==============================================================================

# ------------------------------- SETTINGS (edit here, then re-run) -------------------------------
ALLOWED_LAND_COVER = {10, 20, 30, 40, 60}   # tree, shrub, grass, crop, bare. Built-up (50) and water (80) are excluded
MAX_TOP_SITES = 20                          # how many sites the spaced shortlist contains
TOP_SITES_SPACING_KM = 5.0                  # minimum distance between two shortlist sites

CENTER_LAT = None                           # None = centre of the dataset (median of all cells)
CENTER_LON = None                           # or type your own, e.g. Baghdad city centre is about 33.3152, 44.3661
MIN_DISTANCE_OPTIONS_KM = [0, 20, 30, 40]   # dropdown on the map: shortlist only from cells at least this far from the centre
MAX_DISTANCE_KM = None                      # None = up to the edge of the study area; or a number, e.g. 45
# --------------------------------------------------------------------------------------------------

required_map_columns = {
    "cell_id", "latitude", "longitude", "land_cover_code", "rank",
    "final_priority", "candidate_group", "land_cover_name", "cluster_label",
    "greening_potential_category", "data_confidence",
    "recent_mean_ndvi", "recent_ndvi_change", "seasonal_anomaly",
    "short_reason", "data_uncertainty"
}
map_df = cell_features.reset_index().copy()          # one row per cell, cell_id becomes a normal column
missing = required_map_columns.difference(map_df.columns)
if missing:
    raise ValueError(f"Required model columns missing: {sorted(missing)}. Run the main notebook first.")
if map_df["cell_id"].duplicated().any():
    raise ValueError("Each cell_id must appear exactly once.")
if map_df["rank"].isna().any():
    raise ValueError("Screening ranks are incomplete.")

# 1. Land-cover policy. These are preliminary exclusions, NOT confirmed planting suitability.
map_df["eligible_for_review"] = map_df["land_cover_code"].isin(ALLOWED_LAND_COVER)
# np.select: the first matching condition gives the note; "default" is used when none matches.
map_df["review_note"] = np.select(
    [map_df["land_cover_code"].eq(40), map_df["land_cover_code"].eq(10),
     map_df["land_cover_code"].eq(50), map_df["land_cover_code"].eq(80)],
    ["Cropland: check farming rights and existing use",
     "Existing tree cover: may need maintenance rather than planting",
     "Built-up at sample point: excluded from direct planting screening",
     "Water at sample point: excluded from direct planting screening"],
    default="Preliminary screening only: inspect land use and water access on site")

# 2. Distance of every cell from the centre, in km.
# We convert degrees to metres on a small flat map (error far below 1% for an area this size).
EARTH_RADIUS_M = 6371000.0
center_lat = map_df["latitude"].median() if CENTER_LAT is None else CENTER_LAT
center_lon = map_df["longitude"].median() if CENTER_LON is None else CENTER_LON
map_df["x_m"] = np.radians(map_df["longitude"] - center_lon) * np.cos(np.radians(center_lat)) * EARTH_RADIUS_M
map_df["y_m"] = np.radians(map_df["latitude"] - center_lat) * EARTH_RADIUS_M
map_df["dist_to_center_km"] = (np.hypot(map_df["x_m"], map_df["y_m"]) / 1000).round(2)

# 3. Review pool: eligible cells with enough evidence, best rank first.
review_pool = (map_df.loc[map_df["eligible_for_review"] & map_df["final_priority"].ne("Insufficient evidence")]
               .sort_values(["rank", "cell_id"]).copy())
top100_ids = review_pool.head(100)["cell_id"]
map_df["is_top100"] = map_df["cell_id"].isin(top100_ids)


def pick_spaced_sites(pool, min_spacing_km, max_sites):
    # `pool` is already sorted best-first. We walk down it and keep a cell only if it is at least
    # `min_spacing_km` away from every cell we already kept (a "greedy" selection).
    kept_index, kept_x, kept_y = [], [], []
    for row in pool.itertuples():                      # itertuples: simple, fast row-by-row loop
        too_close = any(np.hypot(row.x_m - x, row.y_m - y) < min_spacing_km * 1000 - 1e-6
                        for x, y in zip(kept_x, kept_y))
        if not too_close:
            kept_index.append(row.Index)
            kept_x.append(row.x_m)
            kept_y.append(row.y_m)
        if len(kept_index) >= max_sites:
            break
    return pool.loc[kept_index]


# 4. One shortlist per distance option (the map dropdown switches between them).
shortlists_by_min_distance = {}
for min_km in MIN_DISTANCE_OPTIONS_KM:
    in_band = review_pool["dist_to_center_km"] >= min_km
    if MAX_DISTANCE_KM is not None:
        in_band = in_band & (review_pool["dist_to_center_km"] <= MAX_DISTANCE_KM)
    shortlists_by_min_distance[min_km] = pick_spaced_sites(review_pool[in_band], TOP_SITES_SPACING_KM, MAX_TOP_SITES)

# Safety checks: no excluded land cover, no duplicates, spacing and distance rules respected.
for min_km, sites in shortlists_by_min_distance.items():
    assert not sites["land_cover_code"].isin([50, 80]).any(), "excluded land cover found in a shortlist"
    assert sites["cell_id"].is_unique, "duplicated site in a shortlist"
    assert (sites["dist_to_center_km"] >= min_km).all(), "a site is closer to the centre than allowed"
    if len(sites) > 1:
        xy = sites[["x_m", "y_m"]].to_numpy()
        gaps = np.hypot(xy[:, None, 0] - xy[None, :, 0], xy[:, None, 1] - xy[None, :, 1]) + np.eye(len(xy)) * 1e12
        assert gaps.min() >= TOP_SITES_SPACING_KM * 1000 - 1e-6, "two sites are closer than the spacing rule"

default_sites = shortlists_by_min_distance[MIN_DISTANCE_OPTIONS_KM[0]]       # first option = default view
map_df["is_spaced_top20"] = map_df["cell_id"].isin(default_sites["cell_id"])

print(f"Total {len(map_df):,} cells | eligible for preliminary review: {int(map_df['eligible_for_review'].sum()):,} "
      f"| excluded (built-up/water/other): {int((~map_df['eligible_for_review']).sum()):,}")
print(f"Distance centre: {center_lat:.4f} N, {center_lon:.4f} E | farthest cell: {map_df['dist_to_center_km'].max():.1f} km")
for min_km, sites in shortlists_by_min_distance.items():
    far_edge = "edge of area" if MAX_DISTANCE_KM is None else f"{MAX_DISTANCE_KM:g} km"
    print(f"  shortlist from >= {min_km:g} km to {far_edge}: {len(sites)} sites "
          f"(>= {TOP_SITES_SPACING_KM:g} km apart, mean distance {sites['dist_to_center_km'].mean():.1f} km)")

# 5. Export files for Streamlit / later use.
export_fields = [
    "cell_id", "latitude", "longitude", "rank", "land_cover_code", "land_cover_name",
    "eligible_for_review", "final_priority", "candidate_group", "cluster_label",
    "greening_potential_category", "recent_mean_ndvi", "recent_ndvi_change",
    "seasonal_anomaly", "data_confidence", "dist_to_center_km", "is_top100", "is_spaced_top20",
    "short_reason", "data_uncertainty", "review_note"
]


def export_table(ids):
    # Always take the rows from map_df, so every table has exactly the same columns.
    return map_df.loc[map_df["cell_id"].isin(ids), export_fields].sort_values("rank")


map_df[export_fields].to_csv("greensite_baghdad_screening_map_data.csv", index=False)
export_table(default_sites["cell_id"]).to_csv("greensite_baghdad_spatial_shortlist.csv", index=False)
export_table(top100_ids).to_csv("greensite_baghdad_top100.csv", index=False)

# One file with every distance option (column min_distance_km tells which option a row belongs to).
all_band_tables = []
for min_km, sites in shortlists_by_min_distance.items():
    band_table = export_table(sites["cell_id"])
    band_table.insert(0, "min_distance_km", min_km)
    all_band_tables.append(band_table)
pd.concat(all_band_tables).to_csv("greensite_baghdad_shortlists_by_distance.csv", index=False)
print("Saved 4 CSV files: screening_map_data, spatial_shortlist, shortlists_by_distance, top100.")


In [ ]:
# ==============================================================================
# Step 10B — Interactive Bokeh map (works in Colab)
# Run after Step 10A. Expected result: a map in the notebook + an HTML file you can download.
# Needs internet only to draw the street / satellite tiles.
# ==============================================================================
from html import escape
from pathlib import Path
from bokeh.plotting import figure, show, output_notebook, save
from bokeh.models import (ColumnDataSource, HoverTool, CustomJS, Div, Select, WMTSTileSource,
                          CDSView, GroupFilter, Legend, LegendItem)
from bokeh.layouts import row, column
from bokeh.resources import CDN

output_notebook()          # makes show() draw the map inside the notebook

# 1. Web Mercator coordinates: the map tiles use metres, not degrees.
k = 6378137.0
def to_mercator(lon, lat):
    return lon * (k * np.pi / 180.0), np.log(np.tan((90.0 + lat) * np.pi / 360.0)) * k

map_df["merc_x"], map_df["merc_y"] = to_mercator(map_df["longitude"], map_df["latitude"])
center_x, center_y = to_mercator(center_lon, center_lat)

# 2. Text shown in the legend, tooltips and the inspector panel.
map_df["priority_display"] = np.select(
    [~map_df["eligible_for_review"], map_df["final_priority"].eq("Insufficient evidence")],
    ["Excluded (built-up / water)", "Insufficient evidence"],
    default=map_df["final_priority"] + " priority")
map_df["cluster_display"] = map_df["cluster_label"].fillna("Not clustered")
map_df["potential_display"] = map_df["greening_potential_category"].fillna("Needs further assessment")


def number_text(series, number_format):
    # Turn numbers into text; empty values become "n/a".
    return series.map(number_format.format).where(series.notna(), "n/a")


map_df["lat_str"] = number_text(map_df["latitude"], "{:.5f}° N")
map_df["lon_str"] = number_text(map_df["longitude"], "{:.5f}° E")
map_df["ndvi_str"] = number_text(map_df["recent_mean_ndvi"], "{:.3f}")
map_df["change_str"] = number_text(map_df["recent_ndvi_change"], "{:+.3f}")
map_df["anom_str"] = number_text(map_df["seasonal_anomaly"], "{:+.3f}")
map_df["dist_str"] = number_text(map_df["dist_to_center_km"], "{:.1f} km")
map_df["gmaps_url"] = ("https://www.google.com/maps/search/?api=1&query="
                       + map_df["latitude"].astype(str) + "," + map_df["longitude"].astype(str))

# Text that is inserted into HTML is escaped, so special characters can never break the panel.
for name in ["candidate_group", "land_cover_name", "short_reason", "data_uncertainty", "review_note", "data_confidence"]:
    map_df[name] = map_df[name].fillna("n/a").map(lambda value: escape(str(value), quote=True))

cols_for_bokeh = [
    "merc_x", "merc_y", "cell_id", "rank", "latitude", "longitude", "lat_str", "lon_str",
    "priority_display", "cluster_display", "potential_display",
    "final_priority", "candidate_group", "land_cover_name", "ndvi_str", "change_str",
    "anom_str", "dist_str", "data_confidence", "short_reason", "data_uncertainty",
    "review_note", "gmaps_url"
]

# 3. Data sources: all cells, the top-100 overlay, and the spaced shortlist (one table per distance option).
source_all = ColumnDataSource(map_df[cols_for_bokeh])
source_top100 = ColumnDataSource(map_df.loc[map_df["is_top100"], cols_for_bokeh])

band_sources = []          # one ColumnDataSource per distance option, in the same order as MIN_DISTANCE_OPTIONS_KM
band_options = []          # text shown in the dropdown
for position, (min_km, sites) in enumerate(shortlists_by_min_distance.items()):
    site_rows = map_df.loc[map_df["cell_id"].isin(sites["cell_id"])].sort_values("rank")
    band_sources.append(ColumnDataSource(site_rows[cols_for_bokeh]))
    limit_text = "" if MAX_DISTANCE_KM is None else f" up to {MAX_DISTANCE_KM:g} km"
    name = "All distances" if min_km == 0 and MAX_DISTANCE_KM is None else f"From {min_km:g} km{limit_text}"
    band_options.append((str(position), f"{name}  ({len(site_rows)} sites)"))
source_top_sites = ColumnDataSource(dict(band_sources[0].data))      # the table the map actually draws

# 4. The figure and the two basemaps.
p = figure(
    title="GreenSite AI — Baghdad | real MODIS cell-centre screening (~1 km)",
    width=920, height=680,
    x_axis_type="mercator", y_axis_type="mercator",
    tools="pan,wheel_zoom,box_zoom,reset,tap,save", active_scroll="wheel_zoom")
osm_tile = WMTSTileSource(url="https://tile.openstreetmap.org/{Z}/{X}/{Y}.png",
                          attribution="© OpenStreetMap contributors")
sat_tile = WMTSTileSource(url="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{Z}/{Y}/{X}.jpg",
                          attribution="Tiles © Esri and imagery contributors")
r_osm = p.add_tile(osm_tile)
r_sat = p.add_tile(sat_tile)
r_sat.visible = False                      # start with the street map

# 5. One scatter renderer PER CATEGORY, so a legend click hides only that category.
priority_colors = {"High priority": "#198754", "Medium priority": "#F2AE40", "Low priority": "#e74c3c",
                   "Insufficient evidence": "#9ca3af", "Excluded (built-up / water)": "#34495e"}
potential_colors = {"More promising": "#2ecc71", "Mixed": "#f1c40f", "Less promising": "#e67e22",
                    "Needs further assessment": "#7f8c8d"}
cluster_names_found = sorted(map_df["cluster_display"].unique().tolist())
cluster_palette = ["#e67e22", "#8b5cf6", "#2ecc71", "#2980b9", "#95a5a6"]
cluster_colors = {name: cluster_palette[i % len(cluster_palette)] for i, name in enumerate(cluster_names_found)}

renderers_by_mode = {}
legends_by_mode = {}
all_point_renderers = []
for mode, field_name, palette in [("priority", "priority_display", priority_colors),
                                  ("cluster", "cluster_display", cluster_colors),
                                  ("potential", "potential_display", potential_colors)]:
    renderers, legend_items = [], []
    for label, color in palette.items():
        count = int((map_df[field_name] == label).sum())
        if count == 0:
            continue
        renderer = p.scatter(
            x="merc_x", y="merc_y", source=source_all,
            view=CDSView(filter=GroupFilter(column_name=field_name, group=label)),
            marker="square", size=8, fill_color=color, line_color=None,
            fill_alpha=0.86, visible=(mode == "priority"),
            selection_color="#00ffff", nonselection_alpha=0.40)
        renderers.append(renderer)
        all_point_renderers.append(renderer)
        short_label = label.split(",")[0]          # long cluster names are shortened in the legend
        legend_items.append(LegendItem(label=f"{short_label} ({count:,})", renderers=[renderer]))
    legend = Legend(items=legend_items, location="top_left", title="Click a category to hide/show it",
                    click_policy="hide", background_fill_alpha=0.85, visible=(mode == "priority"))
    p.add_layout(legend, "center")
    renderers_by_mode[mode] = renderers
    legends_by_mode[mode] = legend

# 6. Overlays: top-100 circles (off by default), spaced shortlist stars (on), and the distance centre.
top100_renderer = p.scatter(x="merc_x", y="merc_y", source=source_top100, marker="circle", size=11,
                            fill_color="#00E5FF", fill_alpha=0.20, line_color="#00B8D4", line_width=1.5,
                            visible=False, selection_color="#00ffff")
top_sites_renderer = p.scatter(x="merc_x", y="merc_y", source=source_top_sites, marker="star", size=17,
                               fill_color="#00e5ff", line_color="#000000", line_width=1.6,
                               visible=True, selection_color="#00ffff")
centre_renderer = p.scatter(x=[center_x], y=[center_y], marker="cross", size=18, line_color="#000000", line_width=3)
overlay_legend = Legend(
    items=[LegendItem(label="Top 100 eligible cells", renderers=[top100_renderer]),
           LegendItem(label=f"Spaced shortlist stars (>= {TOP_SITES_SPACING_KM:g} km apart)", renderers=[top_sites_renderer]),
           LegendItem(label="Distance centre", renderers=[centre_renderer])],
    location="bottom_left", click_policy="hide", background_fill_alpha=0.82, title="Overlays (click to toggle)")
p.add_layout(overlay_legend, "center")

p.add_tools(HoverTool(
    renderers=all_point_renderers + [top100_renderer, top_sites_renderer],
    tooltips=[("Cell & rank", "@cell_id | rank @rank"),
              ("Coordinates", "@lat_str @lon_str"),
              ("Distance to centre", "@dist_str"),
              ("Priority", "@priority_display"),
              ("Cluster", "@cluster_display"),
              ("Recent NDVI / change", "@ndvi_str / @change_str"),
              ("Land cover", "@land_cover_name")]))
p.xaxis.axis_label = "Web Mercator easting (m)"
p.yaxis.axis_label = "Web Mercator northing (m)"

# 7. Controls: colour mode, basemap, and the distance option of the shortlist.
layer_select = Select(title="Colour the cells by:", value="priority", width=380,
                      options=[("priority", "Restoration screening priorities"),
                               ("cluster", "K-Means vegetation clusters"),
                               ("potential", "Preliminary soil potential")])
basemap_select = Select(title="Basemap:", value="street", width=240,
                        options=[("street", "OpenStreetMap"), ("satellite", "Esri satellite imagery")])
band_select = Select(title="Shortlist stars: distance from centre", value="0", width=330, options=band_options)

# Switching the colour mode shows one group of renderers and its legend.
layer_select.js_on_change("value", CustomJS(
    args=dict(select=layer_select,
              p_rends=renderers_by_mode["priority"], c_rends=renderers_by_mode["cluster"], s_rends=renderers_by_mode["potential"],
              p_legend=legends_by_mode["priority"], c_legend=legends_by_mode["cluster"], s_legend=legends_by_mode["potential"]),
    code="""
        const mode = select.value;
        for (const r of p_rends) r.visible = (mode === 'priority');
        for (const r of c_rends) r.visible = (mode === 'cluster');
        for (const r of s_rends) r.visible = (mode === 'potential');
        p_legend.visible = (mode === 'priority');
        c_legend.visible = (mode === 'cluster');
        s_legend.visible = (mode === 'potential');
    """))

basemap_select.js_on_change("value", CustomJS(
    args=dict(r_osm=r_osm, r_sat=r_sat, select=basemap_select),
    code="""
        r_osm.visible = select.value === 'street';
        r_sat.visible = select.value === 'satellite';
    """))

# Switching the distance option copies the matching pre-computed shortlist into the table the stars are drawn from.
band_args = dict(target=source_top_sites, select=band_select)
band_js_lines = ["let chosen = band_0;"]
for position, band_source in enumerate(band_sources):
    band_args[f"band_{position}"] = band_source
    band_js_lines.append(f"if (select.value === '{position}') chosen = band_{position};")
band_js_lines.append("target.data = Object.assign({}, chosen.data);")
band_js_lines.append("target.selected.indices = [];")
band_select.js_on_change("value", CustomJS(args=band_args, code="\n".join(band_js_lines)))

# 8. Inspector panel: click any cell or star to read its details.
inspector = Div(width=365, height=680, text="""
<div style='font-family:Arial,sans-serif;padding:16px;background:#f8f9fa;border-radius:12px;border:1px solid #ccd5d6'>
<h3 style='margin-top:0;color:#198754'>GreenSite AI — Cell Inspector</h3>
<p>Click any coloured cell or a star to see its location, NDVI pattern, K-Means cluster, screening reason and a Google Maps link.</p>
<p><b>Colour mode:</b> use the first dropdown. <b>Legend:</b> click one category to hide or show only that category.</p>
<p><b>Stars:</b> the spaced shortlist (best-ranked eligible cells, kept apart from each other). Use the
<i>distance</i> dropdown to keep only stars that are at least 20, 30 or 40 km from the centre (black cross).</p>
<p><b>Cyan circles:</b> the top 100 eligible cells (turn on in the overlay legend).</p>
<hr><small>These are ~1 km MODIS sample centres, not survey-grade planting parcels.
K-Means and the screening rules are NOT validated planting-success predictions.
Satellite and street tiles need an internet connection.</small>
</div>""")

inspector_js = """
    const indices = source.selected.indices;
    if (!indices.length) return;
    const i = indices[0];
    const d = source.data;
    const value = key => d[key][i] ?? 'n/a';
    div.text = `
    <div style='font-family:Arial,sans-serif;padding:16px;background:#fff;border:2px solid #198754;border-radius:12px;font-size:13px;line-height:1.5'>
      <h3 style='margin:0'>Cell ${value('cell_id')} — Rank ${value('rank')}</h3>
      <p style='color:#198754;font-weight:bold'>${value('priority_display')}</p>
      <b>Coordinates:</b> ${value('lat_str')}, ${value('lon_str')}<br>
      <b>Distance to centre:</b> ${value('dist_str')}<br>
      <p><a href='${value('gmaps_url')}' target='_blank' rel='noopener noreferrer'>Inspect in Google Maps ↗️</a></p>
      <b>Land cover:</b> ${value('land_cover_name')}<br>
      <b>K-Means:</b> ${value('cluster_display')}<br>
      <b>Recent NDVI:</b> ${value('ndvi_str')}<br>
      <b>NDVI change:</b> ${value('change_str')}<br>
      <b>Seasonal anomaly:</b> ${value('anom_str')}<br>
      <b>Soil potential:</b> ${value('potential_display')}<br>
      <b>Data confidence:</b> ${value('data_confidence')}<br>
      <hr>
      <b>Land-use note:</b><br>${value('review_note')}<br><br>
      <b>Screening reason:</b><br>${value('short_reason')}<br><br>
      <b>Limitations:</b><br>${value('data_uncertainty')}
    </div>`;
"""
for source in [source_all, source_top100, source_top_sites]:
    source.selected.js_on_change("indices", CustomJS(args=dict(source=source, div=inspector), code=inspector_js))

# Start with an overview of the whole study area. Zoom in for individual cells.
p.x_range.start = float(map_df["merc_x"].min() - 12000)
p.x_range.end = float(map_df["merc_x"].max() + 12000)
p.y_range.start = float(map_df["merc_y"].min() - 12000)
p.y_range.end = float(map_df["merc_y"].max() + 12000)

layout = column(row(layer_select, basemap_select, band_select), row(p, inspector))
MAP_HTML = Path("greensite_baghdad_bokeh_dashboard_fixed.html")
save(layout, filename=str(MAP_HTML), title="GreenSite AI — Baghdad interactive map", resources=CDN)
print("Interactive map saved to", MAP_HTML.resolve(), "(download it from the Files panel)")
from IPython.display import HTML, display
display(HTML(MAP_HTML.read_text(encoding="utf-8")))   # يعرض نفس الملف المحفوظ داخل النوتبوك      # draws the map inside the notebook

try:                       # in Colab: also download the HTML file to your computer
    from google.colab import files
    files.download(str(MAP_HTML))
except ImportError:
    pass
